# Neural Networks & Backpropagation

---

## Table of Contents
1. [Architecture — Layers, Weights, Activations](#1-architecture)
2. [Forward Pass](#2-forward-pass)
3. [Activation Functions](#3-activation-functions)
4. [Loss Functions for Neural Networks](#4-loss-functions)
5. [Backpropagation — The Chain Rule at Scale](#5-backpropagation)
6. [Vanishing & Exploding Gradients](#6-vanishing--exploding-gradients)
7. [Weight Initialisation](#7-weight-initialisation)
8. [Putting It All Together — Training Loop](#8-training-loop)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os

os.makedirs('img', exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f', 'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888', 'text.color': '#eee',
    'grid.color': '#2a2a2a', 'grid.linewidth': 0.8,
    'font.family': 'monospace', 'axes.titlesize': 12, 'axes.labelsize': 11,
})
ACCENT = '#00e5ff'
ORANGE = '#ff6d00'
GREEN  = '#69ff47'
RED    = '#ff4d6d'
PURPLE = '#c77dff'
YELLOW = '#ffd60a'
WHITE  = '#eeeeee'

---
## 1. Architecture

A feedforward neural network is a **composition of linear transformations and non-linearities**.

For a network with $L$ layers, layer $l$ computes:
$$a^{(l)} = f^{(l)}\!\left(W^{(l)} a^{(l-1)} + b^{(l)}\right)$$

where:
- $a^{(l)} \in \mathbb{R}^{n_l}$ — activations at layer $l$ ($n_l$ = number of neurons)
- $W^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$ — weight matrix
- $b^{(l)} \in \mathbb{R}^{n_l}$ — bias vector
- $f^{(l)}$ — activation function (applied element-wise)
- $a^{(0)} = x$ — the input

The **pre-activation** (before applying $f$) is called $z^{(l)}$:
$$z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)}, \qquad a^{(l)} = f^{(l)}(z^{(l)})$$

**Why non-linearities?** Without $f$, a composition of linear layers collapses into a single linear transformation — the network would be no more expressive than logistic regression regardless of depth. Non-linearities are what give neural networks their capacity to approximate any function (Universal Approximation Theorem).

In [ ]:
# --- Network architecture diagram ---
fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(0, 10); ax.set_ylim(0, 8)
ax.axis('off')
fig.suptitle('Feedforward Neural Network Architecture', fontsize=13)

layer_sizes = [3, 4, 4, 2]
layer_names = ['Input\n$x$', 'Hidden\nlayer 1', 'Hidden\nlayer 2', 'Output\n$\\hat{y}$']
layer_colors = [ACCENT, PURPLE, PURPLE, ORANGE]
layer_x = [1.5, 3.5, 6.0, 8.5]

neuron_positions = []
for l, (n, lx) in enumerate(zip(layer_sizes, layer_x)):
    positions = []
    y_start = (8 - (n - 1) * 1.4) / 2
    for i in range(n):
        y = y_start + i * 1.4
        positions.append((lx, y))
    neuron_positions.append(positions)

# Draw connections
for l in range(len(layer_sizes) - 1):
    for (x1, y1) in neuron_positions[l]:
        for (x2, y2) in neuron_positions[l+1]:
            ax.plot([x1, x2], [y1, y2], color='#333', lw=0.8, zorder=1)

# Draw neurons
for l, (positions, color, name) in enumerate(zip(neuron_positions, layer_colors, layer_names)):
    for i, (x, y) in enumerate(positions):
        circle = plt.Circle((x, y), 0.28, color=color, zorder=3, alpha=0.9)
        ax.add_patch(circle)
        if l == 0:
            ax.text(x, y, f'$x_{i+1}$', ha='center', va='center', fontsize=9, color='#111', fontweight='bold')
        elif l == len(layer_sizes)-1:
            ax.text(x, y, f'$\\hat{{y}}_{i+1}$', ha='center', va='center', fontsize=8, color='#111', fontweight='bold')
        else:
            ax.text(x, y, f'$a$', ha='center', va='center', fontsize=9, color='#111', fontweight='bold')
    ax.text(positions[0][0], 0.3, name, ha='center', va='center',
            fontsize=10, color=color)

# Annotate weight matrix
for l in range(1, len(layer_sizes)):
    mid_x = (layer_x[l-1] + layer_x[l]) / 2
    ax.text(mid_x, 7.3, f'$W^{{({l})}}$, $b^{{({l})}}$',
            ha='center', fontsize=11, color=WHITE,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#222', edgecolor='#555'))

# Forward pass annotation
ax.annotate('', xy=(8.0, 4.0), xytext=(2.0, 4.0),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2.5))
ax.text(5.0, 3.5, 'Forward pass', ha='center', color=GREEN, fontsize=10)

ax.annotate('', xy=(2.0, 3.2), xytext=(8.0, 3.2),
            arrowprops=dict(arrowstyle='->', color=RED, lw=2.5, linestyle='dashed'))
ax.text(5.0, 2.7, 'Backward pass (gradients)', ha='center', color=RED, fontsize=10)

plt.tight_layout()
plt.savefig('img/nn01_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Forward Pass

The forward pass computes the output layer by layer. For a network with $L$ layers and input $x$:

$$a^{(0)} = x$$
$$z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)}, \quad l = 1, \ldots, L$$
$$a^{(l)} = f^{(l)}(z^{(l)}), \quad l = 1, \ldots, L$$
$$\hat{y} = a^{(L)}$$

For a batch of $n$ samples, $X \in \mathbb{R}^{n \times p}$ and all operations are matrix multiplications:
$$Z^{(l)} = A^{(l-1)} W^{(l)\top} + b^{(l)} \in \mathbb{R}^{n \times n_l}$$

**What the forward pass does geometrically**: each layer applies a linear transformation (rotation + scaling via $W$) followed by a non-linear warp (via $f$). Stacking these layers progressively distorts the input space until the classes become linearly separable in the final layer.

In [ ]:
# --- Forward pass on a small network: numerical trace ---
np.random.seed(0)

# Tiny network: 2 -> 3 -> 1
W1 = np.array([[ 0.5, -0.3],
               [ 0.8,  0.2],
               [-0.1,  0.9]])
b1 = np.array([0.1, -0.2, 0.0])
W2 = np.array([[0.4, -0.7, 0.3]])
b2 = np.array([0.0])

def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

x_in = np.array([1.5, -0.5])
z1 = W1 @ x_in + b1
a1 = relu(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')
fig.suptitle('Forward Pass — Numerical Trace', fontsize=13)

steps = [
    ('Input $x$',         x_in,         ACCENT),
    ('$z^{(1)} = W^{(1)}x + b^{(1)}$', z1, PURPLE),
    ('$a^{(1)} = \\text{ReLU}(z^{(1)})$', a1, GREEN),
    ('$z^{(2)} = W^{(2)}a^{(1)} + b^{(2)}$', z2, PURPLE),
    ('$\\hat{y} = \\sigma(z^{(2)})$', a2, ORANGE),
]

for i, (label, values, color) in enumerate(steps):
    x_pos = 0.05 + i * 0.19
    ax.text(x_pos + 0.07, 0.88, label, transform=ax.transAxes,
            ha='center', color=color, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a1a', edgecolor=color))
    for j, v in enumerate(values.ravel()):
        y_pos = 0.65 - j * 0.18
        ax.text(x_pos + 0.07, y_pos, f'{v:.3f}',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=11, color=color,
                bbox=dict(boxstyle='round,pad=0.25', facecolor='#222', edgecolor=color, alpha=0.7))
    if i < len(steps) - 1:
        ax.annotate('', xy=(x_pos + 0.19, 0.65),
                    xytext=(x_pos + 0.15, 0.65),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))

plt.tight_layout()
plt.savefig('img/nn02_forward_pass.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Activation Functions

Activation functions must be **non-linear** and ideally **differentiable** (for backprop). Here are the ones you will encounter everywhere.

**Sigmoid** $\sigma(z) = \frac{1}{1+e^{-z}}$
- Output in $(0,1)$ — good for binary output layers
- Derivative: $\sigma'(z) = \sigma(z)(1-\sigma(z))$, max value 0.25
- Problem: **saturates** for large $|z|$ → gradient ≈ 0 → vanishing gradient in deep networks

**Tanh** $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$
- Output in $(-1,1)$ — zero-centred (better than sigmoid for hidden layers)
- Derivative: $1 - \tanh^2(z)$, max value 1
- Still saturates for large $|z|$

**ReLU** $f(z) = \max(0, z)$
- The default activation for hidden layers in modern networks
- Derivative: 1 if $z > 0$, 0 if $z < 0$ — **no vanishing gradient for positive inputs**
- Problem: **dying ReLU** — neurons with $z < 0$ always have zero gradient, can permanently deactivate

**Leaky ReLU** $f(z) = \max(\alpha z, z)$, $\alpha \approx 0.01$
- Fixes dying ReLU by allowing a small gradient for $z < 0$

**GELU** $f(z) = z \cdot \Phi(z)$ where $\Phi$ is the Gaussian CDF
- Used in transformers (BERT, GPT). Smooth approximation to ReLU, performs better in practice on NLP tasks.

In [ ]:
z = np.linspace(-4, 4, 400)

def sigmoid(z):      return 1 / (1 + np.exp(-z))
def d_sigmoid(z):    s = sigmoid(z); return s * (1-s)
def tanh(z):         return np.tanh(z)
def d_tanh(z):       return 1 - np.tanh(z)**2
def relu(z):         return np.maximum(0, z)
def d_relu(z):       return (z > 0).astype(float)
def leaky_relu(z, a=0.01): return np.where(z > 0, z, a*z)
def d_leaky(z, a=0.01):    return np.where(z > 0, 1.0, a)
def gelu(z):
    return 0.5 * z * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3)))
def d_gelu(z):
    return np.gradient(gelu(z), z)

activations = [
    ('Sigmoid',    sigmoid,    d_sigmoid, ACCENT),
    ('Tanh',       tanh,       d_tanh,    PURPLE),
    ('ReLU',       relu,       d_relu,    GREEN),
    ('Leaky ReLU', leaky_relu, d_leaky,   ORANGE),
    ('GELU',       gelu,       d_gelu,    YELLOW),
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Activation Functions and Their Derivatives', fontsize=13)

for col, (name, f, df, color) in enumerate(activations):
    # Function
    ax = axes[0][col]
    ax.plot(z, f(z), color=color, lw=2.5)
    ax.axhline(0, color='#444', lw=0.8)
    ax.axvline(0, color='#444', lw=0.8)
    ax.set_title(name, color=color)
    ax.set_xlabel('$z$'); ax.set_ylabel('$f(z)$') if col == 0 else None
    ax.grid(True); ax.set_xlim(-4,4)

    # Derivative
    ax = axes[1][col]
    ax.plot(z, df(z), color=color, lw=2.5, linestyle='--')
    ax.axhline(0, color='#444', lw=0.8)
    ax.axvline(0, color='#444', lw=0.8)
    ax.set_xlabel('$z$'); ax.set_ylabel("$f'(z)$") if col == 0 else None
    ax.grid(True); ax.set_xlim(-4,4)
    # Annotate max derivative
    max_d = np.max(np.abs(df(z)))
    ax.set_title(f'max$|f\'|$ = {max_d:.2f}', color=color, fontsize=10)

plt.tight_layout()
plt.savefig('img/nn03_activations.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Loss Functions

The choice of output activation and loss function are coupled — they must match the task.

| Task | Output activation | Loss function |
|------|-------------------|---------------|
| Binary classification | Sigmoid | Binary cross-entropy |
| Multiclass classification | Softmax | Categorical cross-entropy |
| Regression | Linear (none) | MSE or MAE |
| Multi-label classification | Sigmoid (per output) | Binary CE per output |

**Why sigmoid + BCE and not sigmoid + MSE?**

With MSE loss and a sigmoid output, the gradient of the loss w.r.t. $z$ contains $\sigma'(z) = \sigma(z)(1-\sigma(z))$. When the network is confidently wrong ($\sigma(z) \approx 0$ or $\approx 1$), this derivative is near 0 — learning stops exactly when you need it most. BCE cancels the sigmoid derivative, giving a clean gradient $(\hat{p} - y)$.

In [ ]:
z_range = np.linspace(-6, 6, 400)
p_hat   = sigmoid(z_range)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Output Activation + Loss Pairing', fontsize=13)

# Gradient of MSE vs BCE through sigmoid
ax = axes[0]
gradient_bce = p_hat - 1          # d(BCE)/dz for y=1
gradient_mse = (p_hat - 1) * p_hat * (1 - p_hat)  # d(MSE)/dz for y=1
ax.plot(z_range, np.abs(gradient_bce), color=GREEN,  lw=2.5, label='|grad BCE| — stays strong')
ax.plot(z_range, np.abs(gradient_mse), color=RED,    lw=2.5, label='|grad MSE| — vanishes at extremes')
ax.set_xlabel('$z = x^\\top\\theta$'); ax.set_ylabel('|Gradient|')
ax.set_title('BCE keeps gradient signal\nMSE loses it when confident')
ax.legend(fontsize=9); ax.grid(True)

# Regression: MSE vs MAE loss shape
ax = axes[1]
errors = np.linspace(-4, 4, 400)
ax.plot(errors, errors**2,        color=ACCENT,  lw=2.5, label='MSE — quadratic')
ax.plot(errors, np.abs(errors),   color=ORANGE, lw=2.5, label='MAE — linear')
ax.plot(errors, np.log(np.cosh(errors)), color=PURPLE, lw=2.5, label='Huber (log-cosh) — smooth MAE')
ax.set_xlabel('Residual $y - \\hat{y}$'); ax.set_ylabel('Loss')
ax.set_title('Regression losses — how they penalise residuals')
ax.legend(fontsize=9); ax.grid(True); ax.set_ylim(0, 8)

# Softmax temperature
ax = axes[2]
scores = np.array([2.0, 1.0, 0.2])
temps  = [0.3, 1.0, 3.0]
colors_t = [ACCENT, GREEN, ORANGE]
x_cls = np.arange(3)
width = 0.25
for i, (T, color) in enumerate(zip(temps, colors_t)):
    probs = np.exp(scores/T) / np.exp(scores/T).sum()
    ax.bar(x_cls + i*width, probs, width, color=color, alpha=0.8, label=f'T={T}')
ax.set_xticks(x_cls + width); ax.set_xticklabels(['Class 0', 'Class 1', 'Class 2'])
ax.set_ylabel('Softmax probability')
ax.set_title('Softmax temperature — sharper at low T')
ax.legend(fontsize=9); ax.grid(True, axis='y')

plt.tight_layout()
plt.savefig('img/nn04_losses.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Backpropagation — The Chain Rule at Scale

Backpropagation is an efficient algorithm to compute $\nabla_\theta \mathcal{L}$ for all parameters simultaneously. It is **not** a learning algorithm — it is a calculus trick. The learning happens in gradient descent.

### The core idea

Define the **error signal** at layer $l$ as:
$$\delta^{(l)} = \frac{\partial \mathcal{L}}{\partial z^{(l)}}$$

For the output layer $L$:
$$\delta^{(L)} = \frac{\partial \mathcal{L}}{\partial a^{(L)}} \odot f'^{(L)}(z^{(L)})$$

For hidden layer $l < L$ (the **backpropagation recurrence**):
$$\delta^{(l)} = \left(W^{(l+1)\top} \delta^{(l+1)}\right) \odot f'^{(l)}(z^{(l)})$$

The gradients for each layer's parameters:
$$\frac{\partial \mathcal{L}}{\partial W^{(l)}} = \delta^{(l)} \left(a^{(l-1)}\right)^\top$$
$$\frac{\partial \mathcal{L}}{\partial b^{(l)}} = \delta^{(l)}$$

### Why it is efficient

A naive implementation would recompute the chain rule from scratch for every parameter. Backprop instead computes $\delta^{(l)}$ **once per layer** and reuses it for all parameters in that layer. This reduces complexity from $\mathcal{O}(|\theta|^2)$ to $\mathcal{O}(|\theta|)$ — linear in the number of parameters.

### The chain rule unrolled

$$\frac{\partial \mathcal{L}}{\partial W^{(1)}} = \underbrace{\frac{\partial \mathcal{L}}{\partial a^{(L)}}}_{\text{output error}} \cdot \underbrace{\frac{\partial a^{(L)}}{\partial z^{(L)}}}_{f'^{(L)}} \cdot \underbrace{\frac{\partial z^{(L)}}{\partial a^{(L-1)}}}_{W^{(L)}} \cdots \underbrace{\frac{\partial z^{(1)}}{\partial W^{(1)}}}_{a^{(0)}}$$

Every layer contributes one factor $W^{(l)}$ and one factor $f'^{(l)}(z^{(l)})$ to the product. This product is what causes vanishing/exploding gradients.

In [ ]:
# --- Backprop from scratch on a 3-layer network ---
np.random.seed(42)

class TinyNet:
    def __init__(self, sizes):
        self.W = [np.random.randn(sizes[i+1], sizes[i]) * 0.5
                  for i in range(len(sizes)-1)]
        self.b = [np.zeros(sizes[i+1]) for i in range(len(sizes)-1)]

    def forward(self, x):
        self.cache = {'a': [x], 'z': []}
        a = x
        for l, (W, b) in enumerate(zip(self.W, self.b)):
            z = W @ a + b
            self.cache['z'].append(z)
            a = sigmoid(z) if l < len(self.W)-1 else z  # linear output
            self.cache['a'].append(a)
        return a

    def backward(self, y):
        L = len(self.W)
        dW = [None] * L
        db = [None] * L
        # MSE loss gradient at output
        delta = self.cache['a'][-1] - y
        for l in reversed(range(L)):
            a_prev = self.cache['a'][l]
            dW[l] = np.outer(delta, a_prev)
            db[l] = delta.copy()
            if l > 0:
                delta = (self.W[l].T @ delta) * d_sigmoid(self.cache['z'][l-1])
        return dW, db

# Train on a simple regression task
net = TinyNet([1, 8, 8, 1])
np.random.seed(1)
x_train = np.linspace(-3, 3, 60)
y_train = np.sin(x_train)

loss_history = []
lr = 0.05
for epoch in range(2000):
    total_loss = 0
    for xi, yi in zip(x_train, y_train):
        pred = net.forward(np.array([xi]))
        total_loss += 0.5 * (pred[0] - yi)**2
        dW, db = net.backward(np.array([yi]))
        for l in range(len(net.W)):
            net.W[l] -= lr * dW[l]
            net.b[l] -= lr * db[l]
    if epoch % 10 == 0:
        loss_history.append(total_loss / len(x_train))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Backpropagation in Practice', fontsize=13)

# Loss curve
ax = axes[0]
ax.plot(np.arange(len(loss_history))*10, loss_history, color=ACCENT, lw=2.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Training loss — network learns $\\sin(x)$')
ax.set_yscale('log'); ax.grid(True)

# Fit quality
ax = axes[1]
x_test = np.linspace(-3, 3, 200)
y_pred = np.array([net.forward(np.array([xi]))[0] for xi in x_test])
ax.plot(x_test, np.sin(x_test), color=WHITE, lw=2, linestyle='--', label='$\\sin(x)$ (true)')
ax.plot(x_test, y_pred,         color=ORANGE, lw=2.5, label='Network prediction')
ax.scatter(x_train[::5], y_train[::5], color=ACCENT, s=25, zorder=5, label='Training samples')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title('Learned function after 2000 epochs')
ax.legend(fontsize=9); ax.grid(True)

# Gradient magnitudes per layer
ax = axes[2]
# Compute gradient norms for a sample
net2 = TinyNet([1, 6, 6, 6, 1])
grad_norms = []
for xi, yi in zip(x_train[:20], y_train[:20]):
    net2.forward(np.array([xi]))
    dW, _ = net2.backward(np.array([yi]))
    if not grad_norms:
        grad_norms = [[] for _ in dW]
    for l, dw in enumerate(dW):
        grad_norms[l].append(np.linalg.norm(dw))

mean_norms = [np.mean(g) for g in grad_norms]
layer_labels = [f'Layer {l+1}' for l in range(len(mean_norms))]
colors_g = [ACCENT, GREEN, ORANGE, RED]
bars = ax.bar(layer_labels, mean_norms, color=colors_g[:len(mean_norms)], alpha=0.8)
for bar, val in zip(bars, mean_norms):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f'{val:.4f}', ha='center', fontsize=10, color='white')
ax.set_ylabel('Mean gradient norm $\\|\\partial\\mathcal{L}/\\partial W^{(l)}\\|$')
ax.set_title('Gradient norm per layer\n(shrinks toward input = vanishing)')
ax.grid(True, axis='y')

plt.tight_layout()
plt.savefig('img/nn05_backprop.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Vanishing & Exploding Gradients

The gradient of the loss w.r.t. the first layer involves a product of $L$ Jacobians:

$$\frac{\partial \mathcal{L}}{\partial W^{(1)}} \propto \prod_{l=2}^{L} W^{(l)} \cdot \prod_{l=1}^{L} f'^{(l)}(z^{(l)})$$

If the average magnitude of these factors is $< 1$, the product $\to 0$ exponentially with depth — **vanishing gradients**.
If $> 1$, the product $\to \infty$ — **exploding gradients**.

**Vanishing** — the signal from early layers cannot propagate backward. Symptoms: loss stops decreasing, early layers don't learn, model behaves as if it has fewer layers.

**Exploding** — weights update by massive steps, loss diverges or goes NaN.

### Solutions

| Problem | Solution | Mechanism |
|---------|----------|----------|
| Vanishing (activation) | Use ReLU | Gradient is 1 for active neurons, doesn't shrink |
| Vanishing (depth) | Residual connections | Skip connections add an identity path: $\frac{\partial}{\partial x}(f(x)+x) = f'(x)+1 \geq 1$ |
| Both | Batch Normalisation | Keeps $z^{(l)}$ in a well-conditioned range, stabilises gradient magnitudes |
| Exploding | Gradient clipping | $g \leftarrow g \cdot \min\left(1, \frac{c}{\|g\|}\right)$ |
| Both | He / Xavier init | Chooses $\text{Var}(W)$ so that variance of activations is preserved forward and backward |

In [ ]:
np.random.seed(0)
L_depth = 20
n_neurons = 64
n_exp = 500  # samples per experiment

def simulate_gradient_flow(activation, W_scale, depth=L_depth):
    """Track activation std through depth."""
    x = np.random.randn(n_exp)
    stds = [np.std(x)]
    for _ in range(depth):
        W = np.random.randn(n_exp, n_exp) * W_scale / np.sqrt(n_exp)
        z = W @ x
        if activation == 'sigmoid': x = sigmoid(z)
        elif activation == 'tanh':  x = np.tanh(z)
        elif activation == 'relu':  x = np.maximum(0, z)
        elif activation == 'linear': x = z
        stds.append(np.std(x))
    return stds

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Vanishing & Exploding Gradients', fontsize=13)

# Signal magnitude through depth for different activations
ax = axes[0]
for act, color, label in [
    ('sigmoid', RED,    'Sigmoid — vanishes'),
    ('tanh',    ORANGE, 'Tanh — vanishes'),
    ('relu',    GREEN,  'ReLU — stable'),
]:
    stds = simulate_gradient_flow(act, W_scale=1.0)
    ax.semilogy(stds, color=color, lw=2.5, label=label)
ax.set_xlabel('Layer depth'); ax.set_ylabel('Std of activations (log)')
ax.set_title('Activation std through 20 layers')
ax.legend(fontsize=9); ax.grid(True)

# Weight scale effect (vanishing vs exploding)
ax = axes[1]
for scale, color, label in [
    (0.5, RED,    'W scale=0.5 — vanishing'),
    (1.0, GREEN,  'W scale=1.0 — stable'),
    (2.0, ORANGE, 'W scale=2.0 — exploding'),
]:
    stds = simulate_gradient_flow('linear', W_scale=scale)
    ax.semilogy(stds, color=color, lw=2.5, label=label)
ax.set_xlabel('Layer depth'); ax.set_ylabel('Std of activations (log)')
ax.set_title('Effect of weight scale on signal propagation')
ax.legend(fontsize=9); ax.grid(True)

# Residual connection effect
ax = axes[2]
depths = np.arange(1, 21)
# Without residual: gradient shrinks as 0.9^depth
grad_no_res  = 0.9**depths
# With residual: gradient = (f'(x) + 1) chain — stays near 1
grad_res = np.clip((0.9 + 1)**depths / (0.9+1)**depths, 0.5, 1.5)  # simplified
# More accurate: show that skip adds a constant path
grad_res_approx = 1 - 0.05 * depths / 20  # slow, controlled decay
ax.plot(depths, grad_no_res,       color=RED,   lw=2.5, label='Without residual — exponential decay')
ax.plot(depths, grad_res_approx,   color=GREEN, lw=2.5, label='With residual — near-constant')
ax.fill_between(depths, 0, grad_no_res, alpha=0.1, color=RED)
ax.fill_between(depths, 0, grad_res_approx, alpha=0.1, color=GREEN)
ax.set_xlabel('Layer depth'); ax.set_ylabel('Relative gradient magnitude')
ax.set_title('Residual connections preserve gradient flow')
ax.legend(fontsize=9); ax.grid(True); ax.set_ylim(0, 1.2)

plt.tight_layout()
plt.savefig('img/nn06_vanishing_exploding.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Weight Initialisation

Initialisation matters because it sets the initial variance of activations and gradients. Bad initialisation causes vanishing or exploding signals from the very first forward pass.

### Xavier / Glorot Initialisation

Designed for **Tanh** and **Sigmoid** activations. Ensures the variance of activations is approximately equal across layers both forward and backward:

$$W^{(l)} \sim \mathcal{N}\!\left(0,\, \frac{2}{n_{l-1} + n_l}\right) \quad \text{or} \quad W^{(l)} \sim \mathcal{U}\!\left(-\sqrt{\frac{6}{n_{l-1}+n_l}},\, \sqrt{\frac{6}{n_{l-1}+n_l}}\right)$$

### He Initialisation

Designed for **ReLU**. ReLU zeros out half the neurons, so the effective fan-in is halved — the variance needs to be doubled:

$$W^{(l)} \sim \mathcal{N}\!\left(0,\, \frac{2}{n_{l-1}}\right)$$

### Why it works

For a layer with $n_{l-1}$ inputs and weights $w \sim \mathcal{N}(0, \sigma^2)$:
$$\text{Var}(z) = n_{l-1} \cdot \text{Var}(w) \cdot \text{Var}(a^{(l-1)})$$

Setting $\text{Var}(w) = 1/n_{l-1}$ (He: $2/n_{l-1}$) keeps $\text{Var}(z) = \text{Var}(a^{(l-1)})$ — the signal neither amplifies nor shrinks.

In [ ]:
np.random.seed(5)
depth_init = 15
n_init = 256

def run_init_experiment(init_type, activation='relu'):
    x = np.random.randn(n_init)
    stds_fwd  = [float(np.std(x))]
    grads     = [1.0]
    for l in range(depth_init):
        fan_in = n_init
        if init_type == 'zeros':   W = np.zeros((n_init, n_init))
        elif init_type == 'small': W = np.random.randn(n_init, n_init) * 0.01
        elif init_type == 'large': W = np.random.randn(n_init, n_init) * 1.0
        elif init_type == 'xavier':W = np.random.randn(n_init, n_init) * np.sqrt(1.0/fan_in)
        elif init_type == 'he':    W = np.random.randn(n_init, n_init) * np.sqrt(2.0/fan_in)
        z = W @ x
        x = np.maximum(0, z) if activation == 'relu' else np.tanh(z)
        stds_fwd.append(float(np.std(x)))
    return stds_fwd

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Weight Initialisation — Effect on Activation Variance', fontsize=13)

# ReLU network
ax = axes[0]
for init, color, label in [
    ('small',  RED,    'Small (0.01) — vanishes'),
    ('large',  ORANGE, 'Large (1.0) — explodes'),
    ('xavier', PURPLE, 'Xavier — designed for tanh'),
    ('he',     GREEN,  'He — designed for ReLU'),
]:
    stds = run_init_experiment(init, 'relu')
    ax.semilogy(stds, color=color, lw=2.5, label=label)
ax.set_xlabel('Layer'); ax.set_ylabel('Std of activations (log scale)')
ax.set_title('ReLU network — He init keeps variance stable')
ax.legend(fontsize=9); ax.grid(True)
ax.axhline(1.0, color='white', lw=1, linestyle=':', alpha=0.5)

# Distribution of activations at layer 10
ax = axes[1]
layer_target = 10
for init, color, label in [('small', RED, 'Small init'), ('he', GREEN, 'He init')]:
    np.random.seed(5)
    x_dist = np.random.randn(n_init)
    for l in range(layer_target):
        fan_in = n_init
        W = (np.random.randn(n_init, n_init) * 0.01 if init == 'small'
             else np.random.randn(n_init, n_init) * np.sqrt(2.0/fan_in))
        x_dist = np.maximum(0, W @ x_dist)
    ax.hist(x_dist, bins=40, alpha=0.6, color=color, density=True, label=f'{label}')
ax.set_xlabel(f'Activation value at layer {layer_target}')
ax.set_ylabel('Density')
ax.set_title(f'Activation distribution at layer {layer_target}\nSmall init collapses to zero')
ax.legend(fontsize=10); ax.grid(True)

plt.tight_layout()
plt.savefig('img/nn07_initialisation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Training Loop

Putting it all together — a complete training loop:

```
Initialise θ (He or Xavier)

For each epoch:
    Shuffle training data
    For each mini-batch B:
        1. FORWARD PASS
           Compute z^(l), a^(l) for l = 1..L
           Compute loss L(θ)

        2. BACKWARD PASS
           Compute δ^(L) at output
           For l = L-1 down to 1:
               δ^(l) = (W^(l+1)ᵀ δ^(l+1)) ⊙ f'(z^(l))
           Compute ∂L/∂W^(l) = δ^(l) (a^(l-1))ᵀ

        3. UPDATE
           θ ← θ - η · ∇L (or Adam step)
```

One full pass over the training set = one **epoch**. One gradient step on a mini-batch = one **iteration**. These are not the same thing.

In [ ]:
# --- Full training: XOR problem (non-linearly separable) ---
np.random.seed(0)

# XOR dataset
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

class MLP:
    def __init__(self, sizes, lr=0.5):
        self.lr = lr
        self.W = [np.random.randn(sizes[i+1], sizes[i]) * np.sqrt(2/sizes[i])
                  for i in range(len(sizes)-1)]
        self.b = [np.zeros(sizes[i+1]) for i in range(len(sizes)-1)]

    def forward(self, x):
        self.cache = {'a': [x], 'z': []}
        a = x
        for l, (W, b) in enumerate(zip(self.W, self.b)):
            z = W @ a + b
            self.cache['z'].append(z)
            a = sigmoid(z)
            self.cache['a'].append(a)
        return a

    def train_step(self, x, y):
        pred = self.forward(x)
        L_val = -y*np.log(pred+1e-9) - (1-y)*np.log(1-pred+1e-9)
        delta = pred - y
        for l in reversed(range(len(self.W))):
            dW = np.outer(delta, self.cache['a'][l])
            db = delta.copy()
            if l > 0:
                delta = (self.W[l].T @ delta) * d_sigmoid(self.cache['z'][l-1])
            self.W[l] -= self.lr * dW
            self.b[l] -= self.lr * db
        return float(np.mean(L_val))

mlp = MLP([2, 4, 1], lr=2.0)
loss_xor = []
for epoch in range(5000):
    epoch_loss = np.mean([mlp.train_step(x, y) for x, y in zip(X_xor, y_xor)])
    loss_xor.append(epoch_loss)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Training Loop — XOR Problem', fontsize=13)

# Loss curve
ax = axes[0]
ax.plot(loss_xor, color=ACCENT, lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.set_title('XOR is not linearly separable\n— needs hidden layer')
ax.set_yscale('log'); ax.grid(True)

# Decision boundary learned
ax = axes[1]
xx_xor, yy_xor = np.meshgrid(np.linspace(-0.5,1.5,200), np.linspace(-0.5,1.5,200))
Z_xor = np.array([mlp.forward(np.array([xi,yi]))[0]
                  for xi, yi in zip(xx_xor.ravel(), yy_xor.ravel())]).reshape(xx_xor.shape)
ax.contourf(xx_xor, yy_xor, Z_xor, levels=50, cmap='RdBu_r', alpha=0.8)
ax.contour(xx_xor,  yy_xor, Z_xor, levels=[0.5], colors='white', linewidths=2.5)
for xi, yi, label in zip(X_xor[:,0], X_xor[:,1], y_xor):
    color = ORANGE if label == 1 else ACCENT
    ax.scatter(xi, yi, color=color, s=200, zorder=5, edgecolors='white', linewidth=1.5)
    ax.text(xi+0.05, yi+0.05, str(int(label)), color='white', fontsize=12, fontweight='bold')
ax.set_title('Learned XOR decision boundary\n(non-linear — impossible for linear model)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$'); ax.grid(True)

# Predictions
ax = axes[2]
preds = [mlp.forward(x)[0] for x in X_xor]
labels_xor = ['(0,0)→0', '(0,1)→1', '(1,0)→1', '(1,1)→0']
colors_pred = [GREEN if abs(p-y)<0.1 else RED for p, y in zip(preds, y_xor)]
bars = ax.bar(labels_xor, preds, color=colors_pred, alpha=0.8)
ax.axhline(0.5, color='white', lw=1.5, linestyle='--', label='threshold 0.5')
for bar, pred, true in zip(bars, preds, y_xor):
    ax.text(bar.get_x()+bar.get_width()/2, pred+0.02,
            f'{pred:.2f}', ha='center', fontsize=11, color='white')
ax.set_ylabel('$\\hat{p}$'); ax.set_ylim(0, 1.15)
ax.set_title('Predictions after training')
ax.legend(fontsize=9); ax.grid(True, axis='y')

plt.tight_layout()
plt.savefig('img/nn08_training_loop.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Key Takeaways

1. A neural network is a **composition of linear transformations and non-linearities**. Without non-linearities, any depth collapses to a single linear map.
2. **Backpropagation is the chain rule**, applied layer by layer from output to input. It computes all gradients in one backward pass — $\mathcal{O}(|\theta|)$, not $\mathcal{O}(|\theta|^2)$.
3. The error signal $\delta^{(l)}$ flows backward through $W^{(l)\top}$ and is gated by $f'^{(l)}(z^{(l)})$. If $f'$ is small (sigmoid saturation), the signal vanishes.
4. **ReLU** solves vanishing gradients for active neurons. **Residual connections** solve it for depth. **Batch normalisation** stabilises the distribution of activations.
5. **He initialisation** for ReLU, **Xavier** for Tanh/Sigmoid — both designed to preserve activation variance forward and gradient variance backward.
6. The loss choice is not cosmetic — **BCE + Sigmoid** cancels the sigmoid derivative, giving clean gradients. **MSE + Sigmoid** saturates exactly when you need gradients most.
7. One epoch ≠ one gradient step. Batch size, learning rate, and number of epochs interact — there is no free lunch in hyperparameter tuning.

---
*Previous: [`03_logistic_regression.ipynb`](./03_logistic_regression.ipynb) · Next: [`05_decision_trees.ipynb`](./05_decision_trees.ipynb)*